In [ ]:
# =====================================================================================
# セル1: ライブラリの準備とデータ読み込み (CH03・CH04対応・最前面ダイアログ版・8列/9列CSV・10列目以降の追加列に対応)
# =====================================================================================
# 04番: 01番と同じ処理・グラフに加えて、10列目以降の列 (例: V01横ばい終点・最大圧力) も
#        1列1シートでグラフにし、同じExcelにまとめる。8列・9列のCSVでは、出力は01番と同じになる

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
import io
import re
import tkinter as tk
from tkinter import filedialog
import numpy as np

# --- ファイルを読み込んでカラム名を変更するための関数 ---
# 戻り値は (データ, 10列目以降の列名のリスト)。8列・9列形式では10列目以降の列はないので、空のリストになる
def load_and_rename_columns(file_path: str) -> tuple[pd.DataFrame, list]:
    # 8列形式 (例: NR_summary_rise_start_results.csv)
    NEW_COLUMN_NAMES = [
        '元ファイル', 'キャビ厚', '射出速度[mm/s]',
        'CH03_立ち上がり時間(s)', 'CH03_立ち上がり圧力(MPa)',
        'CH04_立ち上がり時間(s)', 'CH04_立ち上がり圧力(MPa)',
        'CH03圧力@CH04立ち上がり時(MPa)'
    ]
    # 9列形式 (例: NRall_summary_rise_start_results.csv)
    # 8列目に「立ち上がり時間差(s)」が追加され、最終列「先行列圧力@後発列立ち上がり時(MPa)」を計算用の列名に対応づける
    NEW_COLUMN_NAMES_9COL = NEW_COLUMN_NAMES[:7] + ['立ち上がり時間差(s)', 'CH03圧力@CH04立ち上がり時(MPa)']
    # 10列以上の形式 (例: pp_NRall_summary_rise_max_pressure_results_2.csv) は、先頭9列が次の列名であることを確かめてから読む
    # (列の位置だけで付け直すと、12列の旧キャビ形式などを誤って読んでしまうため)
    SOURCE_COLUMN_NAMES_9COL = [
        '元ファイル', 'キャビティ厚み[mm]', '射出速度[mm/s]',
        'CH03_立ち上がり時間(s)', 'CH03_立ち上がり圧力(MPa)',
        'CH04_立ち上がり時間(s)', 'CH04_立ち上がり圧力(MPa)',
        '立ち上がり時間差(s)', '先行列圧力@後発列立ち上がり時(MPa)'
    ]
    extra_cols = []
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='shift_jis')

    if len(df.columns) == len(NEW_COLUMN_NAMES):
        df.columns = NEW_COLUMN_NAMES
    elif len(df.columns) == len(NEW_COLUMN_NAMES_9COL):
        df.columns = NEW_COLUMN_NAMES_9COL
        print(f"{os.path.basename(file_path)}: 9列形式として読み込みました。")
        # 9列形式ではCH04が先に立ち上がる行が含まれることがある (時間差ΔT34が負になる)
        n_ch04_first = (df['CH04_立ち上がり時間(s)'] < df['CH03_立ち上がり時間(s)']).sum()
        if n_ch04_first > 0:
            print(f"警告: {n_ch04_first}行でCH04がCH03より先に立ち上がっています。これらの行は時間差ΔT34・流速・流量が負になり、樹脂粘度は負(対数グラフではNaN)になります。")
    elif len(df.columns) > len(NEW_COLUMN_NAMES_9COL) and [str(c).strip() for c in df.columns[:9]] == SOURCE_COLUMN_NAMES_9COL:
        # 先頭9列は9列形式と同じ列名に付け直し、10列目以降は元の列名のまま残す
        extra_cols = [str(c).strip() for c in df.columns[9:]]
        df.columns = NEW_COLUMN_NAMES_9COL + extra_cols
        print(f"{os.path.basename(file_path)}: 9列形式＋追加{len(extra_cols)}列として読み込みました。10列目以降: {', '.join(extra_cols)}")
        n_ch04_first = (df['CH04_立ち上がり時間(s)'] < df['CH03_立ち上がり時間(s)']).sum()
        if n_ch04_first > 0:
            print(f"警告: {n_ch04_first}行でCH04がCH03より先に立ち上がっています。これらの行は時間差ΔT34・流速・流量が負になり、樹脂粘度は負(対数グラフではNaN)になります。")
    else:
        print(f"警告: {os.path.basename(file_path)} のカラム数が期待値と異なります。カラム名の変更をスキップします。")
        print(f"  > 期待するカラム数: {len(NEW_COLUMN_NAMES)} または {len(NEW_COLUMN_NAMES_9COL)}、または10列以上で先頭9列が9列形式の列名と同じこと, 実際のカラム数: {len(df.columns)}")
    return df, extra_cols

# --- ステップ1: ファイルの選択と読み込み ---
print("--- ステップ1: ファイル選択と読み込み ---")

# tkinterの設定
root = tk.Tk()
root.withdraw() # メインウィンドウを非表示にする

# ★追加箇所: ダイアログを強制的に最前面（常に手前）に表示させる設定
root.call('wm', 'attributes', '.', '-topmost', True)

file_paths = filedialog.askopenfilenames(
    title="解析したいCSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

if not file_paths:
    print("ファイルが選択されませんでした。処理を中断します。")
else:
    print(f"{len(file_paths)}個のファイルが選択されました。")
    dataframes = {}
    file_path_map = {}
    extra_columns_map = {}   # ファイルごとの10列目以降の列名 (グラフと統計を追加する列)
    
    for path in file_paths:
        file_name = os.path.basename(path)
        dataframes[file_name], extra_columns_map[file_name] = load_and_rename_columns(path)
        file_path_map[file_name] = os.path.dirname(path) 
        
    print("すべてのファイルの読み込みと前処理が完了しました。\n")
    
    # 読み込んだファイルの名前とパスをプリントする
    print("--- 読み込んだファイル一覧 ---")
    for path in file_paths:
        print(f"ファイル名: {os.path.basename(path)}")
        print(f"ファイルパス: {path}")
        print("-" * 50)

In [ ]:
# =====================================================================================
# セル2: 物理量および統計量(線形/対数)の計算 (groupby柔軟対応版・修正済)
# =====================================================================================

# --- 条件ごとの平均・標準偏差を計算し、列として書き込む関数 (セル4の外れ値除外版でも使う) ---
# extra_cols: 10列目以降の列 (追加列)。時間差などと同じく平均・標準偏差を計算する
def add_group_statistics(df, group_cols, extra_cols=()):
    linear_cols = ['時間差ΔT34(s)', '流速V34(mm/s)', '流量Q34(mm³/s)', '圧力差ΔP34(Pa)']
    log_cols = ['せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)']

    for col in linear_cols:
        df[f'{col}_平均'] = df.groupby(group_cols)[col].transform('mean')
        df[f'{col}_標準偏差上'] = df.groupby(group_cols)[col].transform('std')
        df[f'{col}_標準偏差下'] = -df.groupby(group_cols)[col].transform('std')

    for col in log_cols:
        # 幾何平均・幾何標準偏差 (対数空間で計算して戻す)
        # 修正: DataFrameに一時列を追加してgroupbyを正しく機能させる
        df['temp_log_val'] = np.log10(df[col].replace(0, np.nan))
        
        log_mean = df.groupby(group_cols)['temp_log_val'].transform('mean')
        log_std = df.groupby(group_cols)['temp_log_val'].transform('std')
        
        df[f'{col}_幾何平均'] = 10 ** log_mean
        df[f'{col}_幾何標準偏差上'] = 10 ** (log_mean + log_std)
        df[f'{col}_幾何標準偏差下'] = 10 ** (log_mean - log_std)
        
        # 計算が終わった一時列を削除
        df.drop(columns=['temp_log_val'], inplace=True)

    for col in extra_cols:
        df[f'{col}_平均'] = df.groupby(group_cols)[col].transform('mean')
        df[f'{col}_標準偏差上'] = df.groupby(group_cols)[col].transform('std')
        df[f'{col}_標準偏差下'] = -df.groupby(group_cols)[col].transform('std')

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ2: データ処理（全計算）---")

    for key, df in dataframes.items():
        print(f"  計算中: {key}")

        # 前処理: グループ化に使う変数を先に取得
        df['キャビ厚'] = pd.to_numeric(df['キャビ厚'].astype(str).str.replace('mm', ''), errors='coerce')
        df['温度'] = pd.to_numeric(df['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')
        
        group_cols = ['キャビ厚', '温度', '射出速度[mm/s]']

        # --- データブロックの整合性チェック (欠損警告のみ) ---
        for name, group in df.groupby(group_cols):
            if len(group) < 2:
                print(f"  【警告】条件 (厚み{name[0]}mm, {name[1]}℃, {name[2]}mm/s) のデータ数が{len(group)}件です。標準偏差計算がNaNになる可能性があります。")
                
        print("  ✓ データグループの整合性を確認しました。解析を開始します。")
        # ---------------------------------------------------

        distance_L34 = 17.5   
        OUTLIER_THRESHOLD_PERCENT = 10   # 外れ値の判定基準 [%]: 条件ごとの中央値からこれを超えてずれたショットを外れ値とする
        cavity_thickness_numeric = df['キャビ厚']

        # 物理量の計算 (元の差圧計算ロジックを維持)
        df['時間差ΔT34(s)'] = df['CH04_立ち上がり時間(s)'] - df['CH03_立ち上がり時間(s)']
        df['圧力差ΔP34(Pa)'] = (df['CH03圧力@CH04立ち上がり時(MPa)'] - df['CH04_立ち上がり圧力(MPa)']) * 1000000
        df['流速V34(mm/s)'] = distance_L34 / df['時間差ΔT34(s)']
        df['流路断面積(mm²)'] = cavity_thickness_numeric * 40
        df['流量Q34(mm³/s)'] = df['流速V34(mm/s)'] * df['流路断面積(mm²)']
        df['樹脂粘度η34(Pa・s)'] = (df['圧力差ΔP34(Pa)'] * df['流路断面積(mm²)'] * cavity_thickness_numeric**2) / (12 * distance_L34 * df['流量Q34(mm³/s)'])
        df['せん断速度γ34(1/s)'] = (6 * df['流量Q34(mm³/s)']) / (df['流路断面積(mm²)'] * cavity_thickness_numeric)

        # --- 各列の平均・標準偏差の計算 (groupby適用) ---
        add_group_statistics(df, group_cols, extra_columns_map.get(key, []))

        # --- 外れ値の判定 (条件ごとの中央値からのずれ) ---
        # 平均±3σは外れたショット自身が平均と標準偏差を引っ張るため、10ショット程度では検出できない。
        # そこで外れ値に引っ張られにくい中央値を基準にする。データが3件未満の条件は、どれが外れているか決められないため判定しない。
        # 判定は測定値である時間差と圧力差で行い、そこから計算する量にも同じ印を付ける。
        # (樹脂粘度は圧力差×時間差に比例し、両者のずれは同じ向きに出るため、樹脂粘度自身のずれで判定すると普通のショットまで外れ値になる)
        OUTLIER_LABELS = {'時間差ΔT34(s)': '時間差', '流速V34(mm/s)': '流速', '流量Q34(mm³/s)': '流量',
                          '圧力差ΔP34(Pa)': '圧力差', 'せん断速度γ34(1/s)': 'せん断速度', '樹脂粘度η34(Pa・s)': '樹脂粘度'}
        OUTLIER_SOURCES = {'時間差ΔT34(s)': ['時間差', '流速', '流量', 'せん断速度', '樹脂粘度'],
                           '圧力差ΔP34(Pa)': ['圧力差', '樹脂粘度']}
        for col in OUTLIER_LABELS:
            median = df.groupby(group_cols)[col].transform('median')
            df[f'{col}_中央値からのずれ(%)'] = (df[col] - median) / median.abs() * 100
        outlier_flags = pd.DataFrame(False, index=df.index, columns=list(OUTLIER_LABELS.values()))
        for col, affected in OUTLIER_SOURCES.items():
            count = df.groupby(group_cols)[col].transform('count')
            is_outlier = (df[f'{col}_中央値からのずれ(%)'].abs() > OUTLIER_THRESHOLD_PERCENT) & (count >= 3)
            outlier_flags.loc[is_outlier, affected] = True
        # 印が付いた量の名前を「・」でつなぐ (例: '時間差・流速・流量・せん断速度・樹脂粘度')。外れていなければ空欄
        df['外れ値判定'] = outlier_flags.apply(lambda row: '・'.join(row.index[row]), axis=1)

        # --- 10列目以降の列 (追加列) の外れ値の判定: その列自身の値を、上と同じ基準で判定する ---
        # この判定は「外れ値判定」列には入れない (外れ値を除いた版で除く行は、時間差・圧力差の判定だけで決める)。
        # 外れた列の名前を「・」でつないで「追加列の外れ値判定」列に入れる。外れていなければ空欄
        extra_cols = extra_columns_map.get(key, [])
        if extra_cols:
            extra_flags = pd.DataFrame(False, index=df.index, columns=extra_cols)
            for col in extra_cols:
                median = df.groupby(group_cols)[col].transform('median')
                df[f'{col}_中央値からのずれ(%)'] = (df[col] - median) / median.abs() * 100
                count = df.groupby(group_cols)[col].transform('count')
                extra_flags[col] = (df[f'{col}_中央値からのずれ(%)'].abs() > OUTLIER_THRESHOLD_PERCENT) & (count >= 3)
            df['追加列の外れ値判定'] = extra_flags.apply(lambda row: '・'.join(row.index[row]), axis=1)

        dataframes[key] = df
    print("すべての計算が完了しました。")
else:
    print("エラー: ステップ1が実行されていません。")

In [ ]:
# =====================================================================================
# セル3-1: 集約データのCSV書き出し
# =====================================================================================

print("\n--- ステップ3: 集約データのCSV書き出し ---")

if 'dataframes' in locals() and 'file_path_map' in locals():
    for key, df in dataframes.items():
        output_folder = file_path_map[key] 
        group_cols = ['キャビ厚', '温度', '射出速度[mm/s]']
        
        columns_to_aggregate = ['キャビ厚', '温度', '射出速度[mm/s]']
        for col in ['時間差ΔT34(s)', '流速V34(mm/s)', '流量Q34(mm³/s)', '圧力差ΔP34(Pa)']:
            columns_to_aggregate.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])
        for col in ['せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)']:
            columns_to_aggregate.extend([f'{col}_幾何平均', f'{col}_幾何標準偏差上', f'{col}_幾何標準偏差下'])
        # 10列目以降の列 (追加列) の平均・標準偏差
        for col in extra_columns_map.get(key, []):
            columns_to_aggregate.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])
        
        # 条件グループごとに1行だけ抽出
        aggregated_df = df.drop_duplicates(subset=group_cols)[columns_to_aggregate].reset_index(drop=True)
        
        base_name = os.path.splitext(key)[0]
        output_filename_leaf = f"03_aggregated_summary_{base_name}.csv"
        output_full_path = os.path.join(output_folder, output_filename_leaf)
        
        try:
            aggregated_df.to_csv(output_full_path, index=False, encoding='utf-8-sig')
            print(f"  完了: {output_full_path} に保存しました。") 
        except Exception as e:
            print(f"  エラー: {key} のCSV保存に失敗しました。 {e}")

In [ ]:
# =====================================================================================
# セル3-2: 計算結果の途中確認
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ3: 最終データの確認 ---")
    first_df_key = list(dataframes.keys())[0]
    final_df = dataframes[first_df_key]

    if len(final_df.columns) > 10:
        display_df = pd.concat([final_df.iloc[:, :4], final_df.iloc[:, -4:]], axis=1)
        print("\n【データの最初の4列と最後の4列】")
        print(display_df.head())
    else:
        print("エラー：計算が正常に完了したか確認してください。")

In [ ]:
# =====================================================================================
# セル4: 計算済みデータのCSV保存
# =====================================================================================


if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ4: 計算結果のCSVファイル保存 ---")
    output_folder = os.path.dirname(file_paths[0])
    
    for original_filename, df_to_save in dataframes.items():
        base_name = os.path.splitext(original_filename)[0]
        new_filename = f"01_processed_{base_name}.csv"
        output_path = os.path.join(output_folder, new_filename)
        try:
            df_to_save.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"✅ 保存成功: {output_path}")
        except Exception as e:
            print(f"❌ 保存失敗: {output_path} - {e}")

        # --- 外れ値の行を除いた版 (平均・標準偏差は残ったショットで計算し直す。外れ値の判定はやり直さない) ---
        # 除く行は「外れ値判定」(時間差・圧力差) だけで決める。「追加列の外れ値判定」は判定したときの値のまま残す
        df_removed = df_to_save[df_to_save['外れ値判定'] == ''].copy()
        add_group_statistics(df_removed, ['キャビ厚', '温度', '射出速度[mm/s]'], extra_columns_map.get(original_filename, []))
        removed_path = os.path.join(output_folder, f"02_processed_outliers_removed_{base_name}.csv")
        try:
            df_removed.to_csv(removed_path, index=False, encoding='utf-8-sig')
            print(f"✅ 保存成功: {removed_path} (外れ値 {len(df_to_save) - len(df_removed)}行を除外、残り {len(df_removed)}行)")
        except Exception as e:
            print(f"❌ 保存失敗: {removed_path} - {e}")

In [ ]:
# =====================================================================================
# セル5: 【設定】グラフのレイアウトと軸範囲
# =====================================================================================

CV_THICKNESS_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*'] 
TEMP_MARKERS = {'190℃': 'o', '210℃': 's', '230℃': '^'}
HORIZONTAL_GRAPH_GAP = 10

FIGSIZE_NORMAL = (11, 6)        
FIGSIZE_LOGLOG = (12, 8)        
FIGSIZE_SUMMARY = (12, 6)       
FIGSIZE_LOGLOG_SUMMARY = (12, 8)

LINE_MARKER_SIZE = 10      
SCATTER_MARKER_SIZE = 90  

# 個別グラフ(エラーバー付き)に各ショットを灰色の点で重ね、外れ値を赤×とファイル番号で示す (False で従来のグラフ)
SHOW_SHOTS_IN_GRAPHS = True

MANUAL_AXIS_LIMITS = {
    '射出速度と時間差の関係': {
    # 2026-10-03: 縦軸を0〜0.6 → 0〜1.2 sに変更（1.5mm・2.0mmの低速条件が範囲外になっていたため）
             0: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             1: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             2: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             3: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
     'all_data': {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
    },
    '射出速度と流速の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 5000), 'xinterval': 5, 'yinterval': 500},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 5000), 'xinterval': 5, 'yinterval': 500},
    },
    '射出速度と流量の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 300000), 'xinterval': 5, 'yinterval': 50000},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 300000), 'xinterval': 5, 'yinterval': 50000},
    },
    '射出速度と圧力差の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 3 * 10**7), 'xinterval': 5, 'yinterval': 0.5 * 10**7},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 3*10**7), 'xinterval': 5, 'yinterval': 0.5*10**7},
    },
    'せん断速度と樹脂粘度の関係': {
        # 2026-10-03: 01〜05で統一。縦軸下限を0.1に変更（0.5mmの高速条件が1 Pa・sを下回るため）
        'default': {'xlim': (10**0, 10**5), 'ylim': (10**-1, 3 * 10**4), 'xinterval': None, 'yinterval': None},
        'all_data': {'xlim': (10**0, 10**5), 'ylim': (10**-1, 3 * 10**4), 'xinterval': None, 'yinterval': None},
    }
}

# --- 10列目以降の列 (追加列) のシート名と軸範囲 ---
# シート名はExcelの上限 (31文字) に収まるように短くしている。
# ここにない列は「射出速度と{列名}の関係」を31文字で切った名前になり、縦軸は自動になる
EXTRA_COLUMN_SHEET_NAMES = {
    # V01横ばい終点 (13列形式)
    'V01横ばい終点時間(s)': '射出速度とV01横ばい終点時間の関係',
    'CH03圧力@V01横ばい終点(MPa)': '射出速度とCH03圧力@V01終点の関係',
    'CH04圧力@V01横ばい終点(MPa)': '射出速度とCH04圧力@V01終点の関係',
    '先行列-後発列圧力差@V01横ばい終点(MPa)': '射出速度と圧力差@V01終点の関係',
    # 最大値 (15列形式)
    'CH03_最大圧力(MPa)': '射出速度とCH03最大圧力の関係',
    'CH03_最大圧力時間(s)': '射出速度とCH03最大圧力時間の関係',
    'CH04_最大圧力(MPa)': '射出速度とCH04最大圧力の関係',
    'CH04_最大圧力時間(s)': '射出速度とCH04最大圧力時間の関係',
    '最大圧力差(MPa)': '射出速度と最大圧力差の関係',
    '先行列圧力@後発列最大時(MPa)': '射出速度と先行列圧力@後発列最大時の関係',
}
# 時間の列 (s) は記録開始からの時刻。圧力の列 (MPa) は、pp最大値版0.5mmの値 (最大圧力差が43〜62 MPa) も入るように 0〜70 MPa にしている
EXTRA_TIME_AXIS = {'xlim': (0, 350), 'ylim': (0, 8), 'xinterval': 5, 'yinterval': 1}
EXTRA_PRESSURE_AXIS = {'xlim': (0, 350), 'ylim': (0, 70), 'xinterval': 5, 'yinterval': 10}
for extra_col, extra_sheet in EXTRA_COLUMN_SHEET_NAMES.items():
    extra_axis = EXTRA_TIME_AXIS if extra_col.endswith('(s)') else EXTRA_PRESSURE_AXIS
    MANUAL_AXIS_LIMITS[extra_sheet] = {'default': extra_axis, 'all_data': extra_axis}

def get_extra_sheet_name(col):
    if col in EXTRA_COLUMN_SHEET_NAMES:
        return EXTRA_COLUMN_SHEET_NAMES[col]
    return re.sub(r'[\[\]:*?/\\]', '_', f'射出速度と{col}の関係')[:31]

print("グラフの全体スタイル設定を読み込みました。")

In [ ]:
# =====================================================================================
# セル6・7: 【設定】総合比較グラフのスタイルおよびグラデーション設定
# =====================================================================================

CAVITY_THICKNESS_GRADIENT_SETTINGS = {
    0.5: {'start_color': 'red', 'end_color': 'pink'}, 
    1.0: {'start_color': 'blue', 'end_color': 'lightblue'}, 
    1.5: {'start_color': 'gold', 'end_color': 'yellow'}, 
    2.0: {'start_color': 'green', 'end_color': 'lightgreen'}, 
}

print("総合比較グラフのスタイル設定を読み込みました。")

In [ ]:
# =====================================================================================
# セル8: グラフ作成ツールの定義（個別/総合）
# =====================================================================================

print("\n--- グラフ作成の準備 ---")
plt.rcParams['xtick.direction'] = 'in'; plt.rcParams['ytick.direction'] = 'in'

def set_japanese_font():
    standard_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'IPAexGothic']
    for font in standard_fonts:
        try:
            plt.rcParams['font.family'] = font; plt.rcParams['axes.unicode_minus'] = False; return
        except: continue
set_japanese_font()

class GraphCounter:
    def __init__(self): self.count = 1
    def get_next(self): num = self.count; self.count += 1; return num

def create_safe_filename(prefix, suffix, extension="png"):
    return f"{re.sub(r'[\\/:*?\"<>|]', '_', prefix)}_{re.sub(r'[\\/:*?\"<>|]', '_', suffix)}.{extension}"

def generate_color_gradient(start_color, end_color, num_steps):
    from matplotlib.colors import to_rgba
    if num_steps <= 1: return [to_rgba(start_color)]
    start_rgba = np.array(to_rgba(start_color)); end_rgba = np.array(to_rgba(end_color))
    return [tuple(start_rgba + (end_rgba - start_rgba) * i / (num_steps - 1)) for i in range(num_steps)]

def set_manual_log_ticks(ax, xlims, ylims):
    if xlims and xlims[0] > 0 and xlims[1] > 0:
        ticks_x = [10**i for i in range(int(np.floor(np.log10(xlims[0]))), int(np.ceil(np.log10(xlims[1]))) + 1)]
        ax.set_xticks(ticks_x)
    if ylims and ylims[0] > 0 and ylims[1] > 0:
        ticks_y = [10**i for i in range(int(np.floor(np.log10(ylims[0]))), int(np.ceil(np.log10(ylims[1]))) + 1)]
        ax.set_yticks(ticks_y)

# --- 各ショットの重ね描き（外れ値はファイル番号付きで強調） ---
def plot_shots_with_outliers(ax, df_temp, x_col, y_col, judged_cols):
    if not SHOW_SHOTS_IN_GRAPHS or '外れ値判定' not in df_temp.columns: return
    ax.scatter(df_temp[x_col], df_temp[y_col], s=15, color='gray', alpha=0.5, zorder=1, label='各ショット')
    if all(c in OUTLIER_LABELS for c in judged_cols):
        judged_labels = [OUTLIER_LABELS[c] for c in judged_cols]
        is_outlier = df_temp['外れ値判定'].fillna('').str.split('・').apply(lambda names: any(l in names for l in judged_labels))
        outlier_label = f'外れ値（時間差・圧力差が中央値から±{OUTLIER_THRESHOLD_PERCENT}%超）'
    else:
        # 10列目以降の列 (追加列): その列自身の判定 (「追加列の外れ値判定」列) を使う
        is_outlier = df_temp['追加列の外れ値判定'].fillna('').str.split('・').apply(lambda names: any(c in names for c in judged_cols))
        outlier_label = f'外れ値（この列の値が中央値から±{OUTLIER_THRESHOLD_PERCENT}%超）'
    outliers = df_temp[is_outlier]
    if outliers.empty: return
    ax.scatter(outliers[x_col], outliers[y_col], marker='x', s=60, color='red', zorder=4, label=outlier_label)
    # 同じ条件(射出速度)の外れ値はラベルを1つにまとめ、最も上の点に付ける (例: '079, 080')
    for _, group in outliers.groupby('射出速度[mm/s]'):
        top = group.loc[group[y_col].idxmax()]
        shot_ids = group['元ファイル'].str.extract(r'^(\d+)')[0].fillna(group['元ファイル'])
        ax.annotate(', '.join(shot_ids), (top[x_col], top[y_col]), xytext=(4, 4), textcoords='offset points', fontsize=9, color='red')

# --- 通常グラフ作成関数 ---
def create_linear_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims, ylims, xinterval, yinterval):
    graph_title = f'{y_col}の比較'
    worksheet.set_row(start_row, 294)
    unique_temps = sorted(df_thick['温度'].unique())
    
    # 総合グラフ (エラーバー無)
    fig_sum, ax_sum = plt.subplots(figsize=FIGSIZE_SUMMARY)
    ax_sum.set_title(f"[総合] {graph_title}\n(CV厚: {thickness_label}mm)")
    ax_sum.set_xlabel(x_col); ax_sum.set_ylabel(y_col); ax_sum.grid(True)
    if xlims: ax_sum.set_xlim(xlims); 
    if ylims: ax_sum.set_ylim(ylims)
    if xinterval: 
        ax_sum.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
        ax_sum.tick_params(axis='x', rotation=45)
        
    current_col = 1
    for temp in unique_temps:
        df_temp = df_thick[df_thick['温度'] == temp]
        curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
        
        x = curve_data['射出速度[mm/s]']
        y = curve_data[f'{y_col}_平均']
        yerr = curve_data[f'{y_col}_標準偏差上'] # 絶対値で取得
        color = color_map.get(f"{int(temp)}℃", 'black')
        
        # 総合グラフ描画
        ax_sum.plot(x, y, marker=TEMP_MARKERS.get(f"{int(temp)}℃", 'o'), linestyle='-', color=color, label=f"{int(temp)}℃")
        
        # 個別グラフ (エラーバー有)
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_NORMAL)
        serial_num = counter.get_next()
        ax_ind.errorbar(x, y, yerr=yerr, marker=TEMP_MARKERS.get(f"{int(temp)}℃", 'o'), linestyle='-', color=color, capsize=4, label=f"{int(temp)}℃")
        plot_shots_with_outliers(ax_ind, df_temp, x_col, y_col, [y_col])
        ax_ind.set_title(f"[{serial_num}] 個別: {graph_title}\n(CV厚: {thickness_label}mm / {int(temp)}℃)")
        ax_ind.set_xlabel(x_col); ax_ind.set_ylabel(y_col); ax_ind.grid(True)
        ax_ind.legend(loc='best', frameon=False, fontsize='large')
        if xlims: ax_ind.set_xlim(xlims); 
        if ylims: ax_ind.set_ylim(ylims)
        if xinterval:
            ax_ind.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
            ax_ind.tick_params(axis='x', rotation=45)
        
        fig_ind.tight_layout()
        img_filename = create_safe_filename(f"[{serial_num}]_CV{thickness_label}_ind", f"{y_col}_{int(temp)}C")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
        img_data = io.BytesIO(); fig_ind.savefig(img_data, format='png'); plt.close(fig_ind)
        worksheet.write(start_row - 1, current_col, f"[{serial_num}] {int(temp)}℃ 個別(エラーバー有)")
        worksheet.insert_image(start_row, current_col, '', {'image_data': img_data})
        current_col += 12 + HORIZONTAL_GRAPH_GAP
        
    ax_sum.legend(loc='best', frameon=False, fontsize='large')
    fig_sum.tight_layout()
    serial_num_sum = counter.get_next()
    img_filename_sum = create_safe_filename(f"[{serial_num_sum}]_CV{thickness_label}_sum", y_col)
    fig_sum.savefig(os.path.join(save_folder_path, img_filename_sum), dpi=150)
    img_data_sum = io.BytesIO(); fig_sum.savefig(img_data_sum, format='png'); plt.close(fig_sum)
    worksheet.write(start_row - 1, current_col, f"[{serial_num_sum}] 総合比較(エラーバー無)")
    worksheet.insert_image(start_row, current_col, '', {'image_data': img_data_sum})

# --- 対数グラフ作成関数 ---
def create_loglog_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims, ylims):
    graph_title = f'{y_col}の比較'
    worksheet.set_row(start_row, 375)
    unique_temps = sorted(df_thick['温度'].unique())
    
    fig_sum, ax_sum = plt.subplots(figsize=FIGSIZE_LOGLOG_SUMMARY)
    ax_sum.set_title(f"[総合] {graph_title}\n(CV厚: {thickness_label}mm)")
    ax_sum.set_xlabel('せん断速度 (1/s)'); ax_sum.set_ylabel('樹脂粘度 (Pa・s)'); ax_sum.grid(False)
    ax_sum.set_xscale('log'); ax_sum.set_yscale('log')
    if xlims: ax_sum.set_xlim(xlims)
    if ylims: ax_sum.set_ylim(ylims)
    
    current_col = 1
    for temp in unique_temps:
        df_temp = df_thick[df_thick['温度'] == temp]
        curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
        
        x = curve_data[f'{x_col}_幾何平均']; y = curve_data[f'{y_col}_幾何平均']
        # 幾何標準偏差からエラーバーの距離を算出
        yerr = [y - curve_data[f'{y_col}_幾何標準偏差下'], curve_data[f'{y_col}_幾何標準偏差上'] - y]
        xerr = [x - curve_data[f'{x_col}_幾何標準偏差下'], curve_data[f'{x_col}_幾何標準偏差上'] - x]
        
        color = color_map.get(f"{int(temp)}℃", 'black')
        marker = TEMP_MARKERS.get(f"{int(temp)}℃", 'o')
        
        ax_sum.plot(x, y, marker=marker, linestyle='-', color=color, label=f"{int(temp)}℃")
        
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_LOGLOG)
        serial_num = counter.get_next()
        ax_ind.errorbar(x, y, xerr=xerr, yerr=yerr, marker=marker, linestyle='-', color=color, capsize=4, label=f"{int(temp)}℃")
        plot_shots_with_outliers(ax_ind, df_temp, x_col, y_col, [x_col, y_col])
        ax_ind.set_title(f"[{serial_num}] 個別: {graph_title}\n(CV厚: {thickness_label}mm / {int(temp)}℃)")
        ax_ind.set_xlabel('せん断速度 (1/s)'); ax_ind.set_ylabel('樹脂粘度 (Pa・s)'); ax_ind.grid(False)
        ax_ind.set_xscale('log'); ax_ind.set_yscale('log')
        ax_ind.legend(loc='best', frameon=False, fontsize='large')
        if xlims: ax_ind.set_xlim(xlims)
        if ylims: ax_ind.set_ylim(ylims)
        
        fig_ind.tight_layout()
        img_filename = create_safe_filename(f"[{serial_num}]_CV{thickness_label}_log_ind", f"{y_col}_{int(temp)}C")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
        img_data = io.BytesIO(); fig_ind.savefig(img_data, format='png'); plt.close(fig_ind)
        worksheet.write(start_row - 1, current_col, f"[{serial_num}] {int(temp)}℃ 個別(幾何エラーバー有)")
        worksheet.insert_image(start_row, current_col, '', {'image_data': img_data})
        current_col += 15 + HORIZONTAL_GRAPH_GAP
        
    ax_sum.legend(loc='best', frameon=False, fontsize='large')
    fig_sum.tight_layout()
    serial_num_sum = counter.get_next()
    img_filename_sum = create_safe_filename(f"[{serial_num_sum}]_CV{thickness_label}_log_sum", y_col)
    fig_sum.savefig(os.path.join(save_folder_path, img_filename_sum), dpi=150)
    img_data_sum = io.BytesIO(); fig_sum.savefig(img_data_sum, format='png'); plt.close(fig_sum)
    worksheet.write(start_row - 1, current_col, f"[{serial_num_sum}] 総合比較(エラーバー無)")
    worksheet.insert_image(start_row, current_col, '', {'image_data': img_data_sum})

print("エラーバー対応版のグラフ作成関数を定義しました。")

In [ ]:
# =====================================================================================
# セル9: 全データ総集編グラフ作成（エラーバーなし）
# =====================================================================================

def create_final_all_data_graph(df_full, workbook, worksheet, x_col, y_col, start_row, color_map, counter, save_folder_path, start_col, xlims, ylims, xinterval=None, yinterval=None, is_log=False):
    serial_num = counter.get_next()
    graph_title = f'全データ総合比較: {y_col}'
    fig, ax = plt.subplots(figsize=FIGSIZE_SUMMARY if not is_log else FIGSIZE_LOGLOG_SUMMARY)
    
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for i, thickness_val in enumerate(unique_thicknesses):
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        marker = CV_THICKNESS_MARKERS[i % len(CV_THICKNESS_MARKERS)]
        
        for temp in unique_temps:
            df_temp = df_thick[df_thick['温度'] == temp]
            if df_temp.empty: continue
            curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
            
            if is_log:
                ax.plot(curve_data[f'{x_col}_幾何平均'], curve_data[f'{y_col}_幾何平均'], marker=marker, linestyle='-', color=color_map.get(f"{int(temp)}℃", 'black'))
            else:
                ax.plot(curve_data['射出速度[mm/s]'], curve_data[f'{y_col}_平均'], marker=marker, linestyle='-', color=color_map.get(f"{int(temp)}℃", 'black'))
                
    ax.set_title(f"[{serial_num}] {graph_title}")
    if is_log:
        ax.set_xlabel('せん断速度 (1/s)'); ax.set_ylabel('樹脂粘度 (Pa・s)'); ax.grid(False)
        ax.set_xscale('log'); ax.set_yscale('log')
        set_manual_log_ticks(ax, xlims, ylims)
    else:
        ax.set_xlabel('射出速度[mm/s]'); ax.set_ylabel(y_col); ax.grid(True)
        if xinterval: 
            ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
            ax.tick_params(axis='x', rotation=45)
        if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    
    # 凡例の作成
    from matplotlib.lines import Line2D
    dummy = Line2D([0], [0], marker='none', color='none', linestyle='none')
    temp_h = [Line2D([0], [0], marker='o', color=color_map.get(f"{int(t)}℃", 'k'), linestyle='-') for t in unique_temps]
    thick_h = [Line2D([0], [0], marker=CV_THICKNESS_MARKERS[i%len(CV_THICKNESS_MARKERS)], color='gray', linestyle='None') for i, t in enumerate(unique_thicknesses)]
    legend = ax.legend([dummy]+temp_h+[dummy]+thick_h, ['Temperature']+[f"{int(t)}℃" for t in unique_temps]+['Cavity Thickness']+[f"{t}mm" for t in unique_thicknesses], loc='best', frameon=False)
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness']: text.set_weight('bold')
            
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_all", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    
    worksheet.write(start_row - 1, start_col, f"[{serial_num}] 全データ総合比較(エラーバー無)")
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return 12 if not is_log else 15

In [ ]:
# =====================================================================================
# セル10: Excelレポートとグラフ画像の自動生成
# =====================================================================================

if 'file_paths' in locals() and file_paths:
    print("\n--- Excelレポートの生成を開始します ---")
    output_folder = os.path.dirname(file_paths[0])
    
    for target_key, df in dataframes.items():
        graph_counter = GraphCounter()
        base_name = os.path.splitext(target_key)[0]
        main_graph_folder = os.path.join(output_folder, f"05_graph_images_{base_name}")
        os.makedirs(main_graph_folder, exist_ok=True)
        excel_output_path = os.path.join(output_folder, f"04_graph_output_{base_name}.xlsx")

        with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
            workbook = writer.book
            print(f"対象ファイル: {target_key}")

            unique_temps = sorted(df['温度'].dropna().unique())
            colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'purple']
            color_map = {f"{int(t)}℃": colors[i % len(colors)] for i, t in enumerate(unique_temps)}
            
            sheet_definitions = {
                '射出速度と時間差の関係': ['時間差ΔT34(s)'],
                '射出速度と流速の関係': ['流速V34(mm/s)'],
                '射出速度と流量の関係': ['流量Q34(mm³/s)'],
                '射出速度と圧力差の関係': ['圧力差ΔP34(Pa)']
            }
            log_sheet_def = {
                'せん断速度と樹脂粘度の関係': [('せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)')]
            }
            
            # 10列目以降の列 (追加列) は1列1シートで、時間差などと同じ形式のグラフにする (今の5シートの後ろに付ける)
            extra_sheet_definitions = {}
            for extra_col in extra_columns_map.get(target_key, []):
                extra_sheet = get_extra_sheet_name(extra_col)
                if extra_sheet in extra_sheet_definitions:
                    extra_sheet = f"{extra_sheet[:27]}_{len(extra_sheet_definitions) + 1}"
                extra_sheet_definitions[extra_sheet] = [extra_col]
            linear_sheet_definitions = {**sheet_definitions, **extra_sheet_definitions}
            
            all_sheet_defs = {**sheet_definitions, **log_sheet_def, **extra_sheet_definitions}
            unique_thicknesses = sorted(df['キャビ厚'].dropna().unique())
            
            for sheet_name, cols in all_sheet_defs.items():
                worksheet = workbook.add_worksheet(sheet_name)
                current_row = 1
                current_save_path = os.path.join(main_graph_folder, re.sub(r'[\\/:*?"<>|]', '_', sheet_name))
                os.makedirs(current_save_path, exist_ok=True)
                
                # --- セクション1: キャビティ厚みごとのグラフ（個別エラーバー＋厚み別総合） ---
                for thickness_val, df_thick in df.groupby('キャビ厚'):
                    worksheet.merge_range(current_row, 0, current_row, 40, f"--- キャビティ厚み: {thickness_val}mm ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#DDEBF7'}))
                    current_row += 2
                    
                    chunk_index = unique_thicknesses.index(thickness_val)
                    limits = {}
                    sheet_settings = MANUAL_AXIS_LIMITS.get(sheet_name, {})
                    chunk_specific_settings = sheet_settings.get(chunk_index, sheet_settings.get('default', {}))
                    xlim = chunk_specific_settings.get('xlim')
                    ylim = chunk_specific_settings.get('ylim')
                    
                    if sheet_name in linear_sheet_definitions:
                        for y_col in cols:
                            create_linear_graphs_for_thickness(df_thick, workbook, worksheet, '射出速度[mm/s]', y_col, current_row, color_map, graph_counter, thickness_val, current_save_path, xlim, ylim, chunk_specific_settings.get('xinterval'), chunk_specific_settings.get('yinterval'))
                            current_row += 20
                    elif sheet_name in log_sheet_def:
                        for x_col, y_col in cols:
                            create_loglog_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, thickness_val, current_save_path, xlim, ylim)
                            current_row += 20
                    current_row += 3
                
                # --- セクション2: 全データ総集編グラフ（全厚み・全温度を重ねる／エラーバー無） ---
                worksheet.merge_range(current_row, 0, current_row, 40, "--- 全データ総集編グラフ ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#BFBFBF'}))
                current_row += 2
                final_graph_col = 1
                
                final_limits = MANUAL_AXIS_LIMITS.get(sheet_name, {}).get('all_data', {})
                if sheet_name in linear_sheet_definitions:
                    for y_col in cols:
                        span = create_final_all_data_graph(df, workbook, worksheet, '射出速度[mm/s]', y_col, current_row, color_map, graph_counter, current_save_path, final_graph_col, final_limits.get('xlim'), final_limits.get('ylim'), final_limits.get('xinterval'), final_limits.get('yinterval'), is_log=False)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                elif sheet_name in log_sheet_def:
                    for x_col, y_col in cols:
                        span = create_final_all_data_graph(df, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, current_save_path, final_graph_col, final_limits.get('xlim'), final_limits.get('ylim'), is_log=True)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                        
            print(f"\n--- ファイル '{target_key}' の処理が完了しました ---")